# NB06 results report
Reads every `results/*.json` and builds tables and figures for the report. Computes no new numbers:
everything shown is copied from a JSON that a notebook wrote. Runs on Kaggle or locally from `kaggle/`.

On Kaggle: upload your downloaded `results/` folder as a private dataset `spellspeak-results`.

In [ ]:
# Repo code (per.py, words.json). Kaggle cannot see your laptop, so pick one:
#   A) public GitHub repo + Internet ON in notebook settings -> git clone below
#   B) private repo -> upload the repo folder as a private Kaggle Dataset "spellspeak-code" and add it here
import json, os, platform, subprocess, sys, time
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path

REPO_URL = "https://github.com/ddrlve/SpellSpeak.git"
ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
CANDIDATES = [Path("/kaggle/input/spellspeak-code"), Path.cwd().parent, WORK / "SpellSpeak"]
REPO = next((p for p in CANDIDATES if (p / "backend" / "per.py").exists()), None)
if REPO is None:
    REPO = WORK / "SpellSpeak"
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
sys.path.insert(0, str(REPO / "backend"))
RESULTS = WORK / "results"
RESULTS.mkdir(parents=True, exist_ok=True)
print("repo:", REPO, "| results:", RESULTS)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

SOURCES = [Path("/kaggle/input/spellspeak-results"), REPO / "results", RESULTS]
files = {}
for src in SOURCES:
    for f in sorted(src.glob("*.json")) if src.exists() else []:
        files.setdefault(f.name, f)   # first source wins
results = {name: json.loads(f.read_text(encoding="utf-8")) for name, f in files.items()}
print("found:", list(results))
if not results:
    print("No results yet. Run NB00 to NB04 first. Nothing to report.")

In [ ]:
def flatten(d: dict, prefix: str = "") -> dict:
    out = {}
    for k, v in d.items():
        key = f"{prefix}{k}"
        if isinstance(v, dict):
            out.update(flatten(v, key + "."))
        elif not isinstance(v, list):
            out[key] = v
    return out


rows = []
for name, r in results.items():
    for k, v in flatten(r.get("metrics", {})).items():
        rows.append({"notebook": r.get("notebook", name), "metric": k, "value": v,
                     "device": r.get("device", {}).get("name"), "date": r.get("date"),
                     "n_utterances": r.get("data", {}).get("n_utterances")})
table = pd.DataFrame(rows)
table

In [ ]:
# Figure: PER and WER per noise condition (only if NB04 ran).
nb04 = results.get("NB04_noise_conditions.json")
if nb04:
    m = pd.DataFrame(nb04["metrics"]).T
    ax = m[["per_corpus", "wer_zipformer"]].plot(kind="bar", rot=0, title="Noise conditions (NB04)")
    ax.set_ylabel("error rate")
    plt.tight_layout()
    plt.savefig(RESULTS / "fig_noise.png", dpi=150)
    plt.show()

In [ ]:
# Markdown table for the report. Every row cites its source JSON.
if not table.empty:
    md = table.to_markdown(index=False)
    (RESULTS / "report_tables.md").write_text(md, encoding="utf-8")
    print(md)